# Ablation Study

This notebook presents a systematic investigation of the factors driving
performance differences between the three evaluated models.

Investigation 1: Training Data Volume Compensation
Does increasing the training budget for the cross-viewpoint protocol
recover the performance gap caused by reduced viewpoint coverage?
Models are trained with iteration counts scaled to match the total
image passes of the standard protocol.

Investigation 2: AnomalyDINO Single-Class vs Multi-Class
Does AnomalyDINO's performance improve when evaluated per category
rather than with a global multi-class memory bank?
This directly quantifies the cost of the multi-class setting for
memory-based nearest-neighbour methods.

Investigation 3: Training Compute Equalisation Between Models
Dinomaly and INP-Former use different training conventions
(iterations vs epochs) resulting in a large disparity in total
image passes. This investigation equalises compute budgets
to assess whether performance differences reflect architecture
or training volume.

In [ ]:
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

from google.colab import drive
import sys

drive.mount('/content/drive')

repo_path = '/content/drive/MyDrive/BachelorsThesis'
dataset_root = '/content/drive/MyDrive/datasets/realiad_512'

if not os.path.exists(repo_path):
    !git clone https://github.com/PurpleMono/BachelorsThesis.git {repo_path}
    !git -C {repo_path} submodule update --init
else:
    !git -C {repo_path} pull
    !git -C {repo_path} submodule update --init

!git -C {repo_path}/models/inp_former fetch origin
!git -C {repo_path}/models/inp_former checkout 6041e2b

sys.path.insert(0, repo_path)

!pip install anomalib==2.3.3 ADEval einops timm kornia -q

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import shutil

local_dataset_root = '/content/realiad_512'

if not os.path.exists(local_dataset_root):
    print("Copying Real-IAD from Drive to local storage...")
    shutil.copytree(dataset_root, local_dataset_root)
    print(f"Dataset ready at: {local_dataset_root}")
else:
    print(f"Dataset already on local storage: {local_dataset_root}")

dataset_root = local_dataset_root
print(f"Active dataset root: {dataset_root}")

In [ ]:
import importlib.util
import pandas as pd
import numpy as np
import gc

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

realiad_utils = load_module("realiad_utils", f"{repo_path}/data/realiad_utils.py")
trainer = load_module("trainer", f"{repo_path}/models/trainer.py")
metrics = load_module("metrics", f"{repo_path}/evaluation/metrics.py")

load_realiad_all = realiad_utils.load_realiad_all
get_crossview_split = realiad_utils.get_crossview_split
train_dinomaly = trainer.train_dinomaly
train_anomalydino = trainer.train_anomalydino
train_inpformer = trainer.train_inpformer
run_inference = trainer.run_inference
run_inference_inpformer = trainer.run_inference_inpformer
compute_i_auroc = metrics.compute_i_auroc
compute_s_auroc = metrics.compute_s_auroc
compute_degradation_ratio = metrics.compute_degradation_ratio

print("All modules loaded")

In [ ]:
df = load_realiad_all(data_root=dataset_root)

train_views = ['C1', 'C2']
test_views = ['C3', 'C4', 'C5']

train_df_cv, test_df_cv = get_crossview_split(
    df, train_views=train_views, test_views=test_views)

train_df_cv = train_df_cv[train_df_cv['label'] == 0].reset_index(drop=True)
train_df_std = df[(df['split'] == 'train') & (df['label'] == 0)]
test_df_std = df[df['split'] == 'test']

# Load standard protocol results as baseline
results_dinomaly_std = pd.read_csv(
    f'{repo_path}/results/dinomaly_standard_scores.csv')
results_anomalydino_std = pd.read_csv(
    f'{repo_path}/results/anomalydino_standard_scores.csv')
results_inpformer_std = pd.read_csv(
    f'{repo_path}/results/inpformer_standard_scores.csv')

# Load cross-view results as baseline
results_dinomaly_cv = pd.read_csv(
    f'{repo_path}/results/dinomaly_crossview_scores.csv')
results_anomalydino_cv = pd.read_csv(
    f'{repo_path}/results/anomalydino_crossview_scores.csv')
results_inpformer_cv = pd.read_csv(
    f'{repo_path}/results/inpformer_crossview_scores.csv')

i_auroc_din_std = compute_i_auroc(results_dinomaly_std)
i_auroc_dino_std = compute_i_auroc(results_anomalydino_std)
i_auroc_inp_std = compute_i_auroc(results_inpformer_std)

i_auroc_din_cv = compute_i_auroc(results_dinomaly_cv)
i_auroc_dino_cv = compute_i_auroc(results_anomalydino_cv)
i_auroc_inp_cv = compute_i_auroc(results_inpformer_cv)

print(f"Standard protocol baselines loaded")
print(f"Cross-view baselines loaded")
print(f"\nCross-view training images: {len(train_df_cv)}")
print(f"Standard training images: {len(train_df_std)}")
print(f"Image ratio: {len(train_df_cv)/len(train_df_std):.2f}")

## Investigation 1: Training Data Volume Compensation

The cross-viewpoint protocol reduces training data to approximately 2/5
of the standard protocol. This investigation tests whether scaling the
training budget proportionally recovers the performance gap.

Scaled iteration counts to match total image passes of standard protocol:
- Dinomaly: 125,000 iterations (standard: 50,000)
- INP-Former: 500 epochs (standard: 200)
- AnomalyDINO: sampling ratio 0.25 (proportionally larger memory bank
  to match the relative coverage of the standard protocol)

If performance recovers, the degradation is attributable to training volume.
If it does not recover, viewpoint coverage itself is the primary factor.

In [ ]:
model_din_abl1 = train_dinomaly(
    train_df=train_df_cv,
    n_iterations=125000,
    batch_size=16,
    dropout_rate=0.4,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{repo_path}/results/weights/dinomaly_abl1_volume.pth'
)

results_din_abl1 = run_inference(
    model=model_din_abl1,
    test_df=test_df_cv,
    model_name='Dinomaly',
    device='cuda',
    batch_size=8,
    repo_path=repo_path,
    max_ratio=0.001
)

results_din_abl1.to_csv(
    f'{repo_path}/results/dinomaly_abl1_volume_scores.csv', index=False)

i_auroc_din_abl1 = compute_i_auroc(results_din_abl1)
print(f"Dinomaly I-AUROC (standard):   {i_auroc_din_std:.4f}")
print(f"Dinomaly I-AUROC (cross-view): {i_auroc_din_cv:.4f}")
print(f"Dinomaly I-AUROC (ablation 1): {i_auroc_din_abl1:.4f}")
print(f"Recovery: {i_auroc_din_abl1 - i_auroc_din_cv:.4f}")

torch.cuda.empty_cache()
gc.collect()
del model_din_abl1
print("GPU memory cleared")

In [ ]:
model_dino_abl1 = train_anomalydino(
    train_df=train_df_cv,
    device='cuda',
    repo_path=repo_path,
    sampling_ratio=0.25,
    save_path=f'{repo_path}/results/weights/anomalydino_abl1_volume.pt'
)

results_dino_abl1 = run_inference(
    model=model_dino_abl1,
    test_df=test_df_cv,
    model_name='AnomalyDINO',
    device='cuda',
    batch_size=8,
    repo_path=repo_path
)

results_dino_abl1.to_csv(
    f'{repo_path}/results/anomalydino_abl1_volume_scores.csv', index=False)

i_auroc_dino_abl1 = compute_i_auroc(results_dino_abl1)
print(f"AnomalyDINO I-AUROC (standard):   {i_auroc_dino_std:.4f}")
print(f"AnomalyDINO I-AUROC (cross-view): {i_auroc_dino_cv:.4f}")
print(f"AnomalyDINO I-AUROC (ablation 1): {i_auroc_dino_abl1:.4f}")
print(f"Recovery: {i_auroc_dino_abl1 - i_auroc_dino_cv:.4f}")

torch.cuda.empty_cache()
gc.collect()
del model_dino_abl1
print("GPU memory cleared")

In [ ]:
model_inp_abl1 = train_inpformer(
    train_df=train_df_cv,
    dataset_root=dataset_root,
    n_epochs=500,
    batch_size=16,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{repo_path}/results/weights/inpformer_abl1_volume.pth'
)

results_inp_abl1 = run_inference_inpformer(
    model=model_inp_abl1,
    test_df=test_df_cv,
    dataset_root=dataset_root,
    device='cuda',
    batch_size=8,
    repo_path=repo_path
)

results_inp_abl1.to_csv(
    f'{repo_path}/results/inpformer_abl1_volume_scores.csv', index=False)

i_auroc_inp_abl1 = compute_i_auroc(results_inp_abl1)
print(f"INP-Former I-AUROC (standard):   {i_auroc_inp_std:.4f}")
print(f"INP-Former I-AUROC (cross-view): {i_auroc_inp_cv:.4f}")
print(f"INP-Former I-AUROC (ablation 1): {i_auroc_inp_abl1:.4f}")
print(f"Recovery: {i_auroc_inp_abl1 - i_auroc_inp_cv:.4f}")

torch.cuda.empty_cache()
gc.collect()
del model_inp_abl1
print("GPU memory cleared")

In [ ]:
abl1_summary = pd.DataFrame({
    'Model': ['Dinomaly', 'AnomalyDINO', 'INP-Former'],
    'Standard': [i_auroc_din_std, i_auroc_dino_std, i_auroc_inp_std],
    'Cross-View': [i_auroc_din_cv, i_auroc_dino_cv, i_auroc_inp_cv],
    'Cross-View + Volume': [i_auroc_din_abl1, i_auroc_dino_abl1, i_auroc_inp_abl1],
    'Degradation (%)': [
        compute_degradation_ratio(i_auroc_din_std, i_auroc_din_cv),
        compute_degradation_ratio(i_auroc_dino_std, i_auroc_dino_cv),
        compute_degradation_ratio(i_auroc_inp_std, i_auroc_inp_cv),
    ],
    'Recovery after Volume (delta)': [
        i_auroc_din_abl1 - i_auroc_din_cv,
        i_auroc_dino_abl1 - i_auroc_dino_cv,
        i_auroc_inp_abl1 - i_auroc_inp_cv,
    ],
})

print("=" * 70)
print("INVESTIGATION 1: TRAINING DATA VOLUME COMPENSATION")
print("=" * 70)
print(abl1_summary.round(4).to_string(index=False))
abl1_summary.to_csv(
    f'{repo_path}/results/ablation1_volume_summary.csv', index=False)
print("\nSaved to results/ablation1_volume_summary.csv")

## Investigation 2: AnomalyDINO Single-Class vs Multi-Class

AnomalyDINO was originally designed for the single-class few-shot setting.
This investigation evaluates it per category with a dedicated memory bank
to directly quantify the performance cost of the multi-class setting.

A global multi-class memory bank risks cross-category feature contamination:
anomalous patches from one category may find false nearest-neighbours
in normal features from unrelated categories, reducing detection sensitivity.

Categories evaluated: audiojack, pcb, button_battery, usb, toothbrush
(selected to represent diverse object types and defect characteristics)

In [ ]:
# Select representative categories for single-class evaluation
SINGLECLASS_CATEGORIES = ['audiojack', 'pcb', 'button_battery', 'usb', 'toothbrush']

results_singleclass = {}
results_multiclass_subset = {}

for category in SINGLECLASS_CATEGORIES:
    print(f"\nProcessing {category}...")

    # Load category data
    import importlib.util
    spec = importlib.util.spec_from_file_location(
        "realiad_utils", f"{repo_path}/data/realiad_utils.py")
    ru = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(ru)

    df_cat = ru.load_realiad_category(
        category_root=f'{dataset_root}/{category}',
        json_path=f'{dataset_root}/realiad_jsons/{category}.json'
    )

    train_cat = df_cat[(df_cat['split'] == 'train') & (df_cat['label'] == 0)]
    test_cat = df_cat[df_cat['split'] == 'test']

    # Train single-class AnomalyDINO on this category only
    model_sc = train_anomalydino(
        train_df=train_cat,
        device='cuda',
        repo_path=repo_path,
        sampling_ratio=0.1,
    )

    # Run inference
    results_sc = run_inference(
        model=model_sc,
        test_df=test_cat,
        model_name='AnomalyDINO',
        device='cuda',
        batch_size=8,
        repo_path=repo_path
    )

    i_auroc_sc = compute_i_auroc(results_sc)
    results_singleclass[category] = i_auroc_sc
    print(f"{category} single-class I-AUROC: {i_auroc_sc:.4f}")

    # Get multi-class result for same category from standard protocol
    results_mc_cat = results_anomalydino_std[
        results_anomalydino_std['category'] == category]
    if len(results_mc_cat) > 0:
        i_auroc_mc = compute_i_auroc(results_mc_cat)
        results_multiclass_subset[category] = i_auroc_mc
        print(f"{category} multi-class I-AUROC:  {i_auroc_mc:.4f}")

    torch.cuda.empty_cache()
    gc.collect()
    del model_sc

In [ ]:
abl2_data = []
for cat in SINGLECLASS_CATEGORIES:
    sc = results_singleclass.get(cat, float('nan'))
    mc = results_multiclass_subset.get(cat, float('nan'))
    abl2_data.append({
        'Category': cat,
        'Single-Class I-AUROC': sc,
        'Multi-Class I-AUROC': mc,
        'Delta (single - multi)': sc - mc if not (
            np.isnan(sc) or np.isnan(mc)) else float('nan')
    })

abl2_summary = pd.DataFrame(abl2_data)
abl2_summary.loc['Mean'] = abl2_summary.mean(numeric_only=True)
abl2_summary.at['Mean', 'Category'] = 'Mean'

print("=" * 65)
print("INVESTIGATION 2: ANOMALYDINO SINGLE-CLASS VS MULTI-CLASS")
print("=" * 65)
print(abl2_summary.round(4).to_string(index=False))
abl2_summary.to_csv(
    f'{repo_path}/results/ablation2_singleclass_summary.csv', index=False)
print("\nSaved to results/ablation2_singleclass_summary.csv")

## Investigation 3: Training Compute Equalisation Between Models

Dinomaly uses 50,000 iterations with batch size 16 on Real-IAD,
resulting in approximately 800,000 total image passes.

INP-Former uses 200 epochs on Real-IAD (approximately 36,465 normal images),
resulting in approximately 7,293,000 total image passes.

This represents a roughly 9x disparity in training compute between the two models.
This investigation trains both models with an equalised compute budget
to assess whether observed performance differences reflect architecture
or training volume.

Equalised budget: both models trained for approximately 3,000,000 image passes
- Dinomaly: 187,500 iterations (batch size 16)
- INP-Former: 82 epochs (approximately matching Dinomaly image passes)

In [ ]:
# Compute equalised training: approximately 3,000,000 image passes for both models
# This is the geometric mean between Dinomaly standard (800k) and INP-Former standard (7.3M)
EQUALISED_ITERS_DINOMALY = 187500
EQUALISED_EPOCHS_INPFORMER = 82

# Dinomaly with equalised compute
model_din_abl3 = train_dinomaly(
    train_df=train_df_std,
    n_iterations=EQUALISED_ITERS_DINOMALY,
    batch_size=16,
    dropout_rate=0.4,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{repo_path}/results/weights/dinomaly_abl3_compute.pth'
)

results_din_abl3 = run_inference(
    model=model_din_abl3,
    test_df=test_df_std,
    model_name='Dinomaly',
    device='cuda',
    batch_size=8,
    repo_path=repo_path,
    max_ratio=0.001
)

results_din_abl3.to_csv(
    f'{repo_path}/results/dinomaly_abl3_compute_scores.csv', index=False)
i_auroc_din_abl3 = compute_i_auroc(results_din_abl3)

print(f"Dinomaly standard (50k iter):    {i_auroc_din_std:.4f}")
print(f"Dinomaly equalised (187.5k iter): {i_auroc_din_abl3:.4f}")

torch.cuda.empty_cache()
gc.collect()
del model_din_abl3
print("GPU memory cleared")

In [ ]:
model_inp_abl3 = train_inpformer(
    train_df=train_df_std,
    dataset_root=dataset_root,
    n_epochs=EQUALISED_EPOCHS_INPFORMER,
    batch_size=16,
    device='cuda',
    repo_path=repo_path,
    save_path=f'{repo_path}/results/weights/inpformer_abl3_compute.pth'
)

results_inp_abl3 = run_inference_inpformer(
    model=model_inp_abl3,
    test_df=test_df_std,
    dataset_root=dataset_root,
    device='cuda',
    batch_size=8,
    repo_path=repo_path
)

results_inp_abl3.to_csv(
    f'{repo_path}/results/inpformer_abl3_compute_scores.csv', index=False)
i_auroc_inp_abl3 = compute_i_auroc(results_inp_abl3)

print(f"INP-Former standard (200 epochs):   {i_auroc_inp_std:.4f}")
print(f"INP-Former equalised (82 epochs):    {i_auroc_inp_abl3:.4f}")

torch.cuda.empty_cache()
gc.collect()
del model_inp_abl3
print("GPU memory cleared")

In [ ]:
abl3_summary = pd.DataFrame({
    'Model': ['Dinomaly', 'INP-Former'],
    'Standard I-AUROC': [i_auroc_din_std, i_auroc_inp_std],
    'Standard Image Passes': [800000, 7293000],
    'Equalised I-AUROC': [i_auroc_din_abl3, i_auroc_inp_abl3],
    'Equalised Image Passes': [3000000, 3000000],
    'Delta (equalised - standard)': [
        i_auroc_din_abl3 - i_auroc_din_std,
        i_auroc_inp_abl3 - i_auroc_inp_std,
    ],
})

print("=" * 70)
print("INVESTIGATION 3: TRAINING COMPUTE EQUALISATION")
print("=" * 70)
print(abl3_summary.round(4).to_string(index=False))
abl3_summary.to_csv(
    f'{repo_path}/results/ablation3_compute_summary.csv', index=False)
print("\nSaved to results/ablation3_compute_summary.csv")

## Combined Ablation Summary

Synthesises findings across all three investigations.

In [ ]:
print("=" * 70)
print("ABLATION STUDY COMBINED SUMMARY")
print("=" * 70)

print("\nInvestigation 1: Training Data Volume Compensation")
print(abl1_summary.round(4).to_string(index=False))

print("\nInvestigation 2: AnomalyDINO Single vs Multi-Class")
print(abl2_summary.round(4).to_string(index=False))

print("\nInvestigation 3: Compute Equalisation")
print(abl3_summary.round(4).to_string(index=False))